## 4. Forward PINNs

**Question.** If a small neural network is given only the ODE and the initial condition, how well does it predict the population, compared with a standard solver? All data are *synthetic and dimensionless* ($x=N/K$, $\tau=rt$). The domain is $\tau\in[0,T]$ ($T=10$; for Model A, $T$ is the extinction time) and $x\in[0,1]$.

**Equations.** Model A: $\dot x=G(x)-q$; Model B: $\dot x=G(x)-ex$; $G(x)=x(1-x)(x/a-1)$. Write $\dot x=f(x)$.

**Network with a hard initial condition.** A fully connected network $N(s;\theta)$ with layer sizes $(1,32,32,1)$ (JAX, float64) and the *swish* activation $z/(1+e^{-z})$ (selected from tanh, swish and sigmoid, see below), with fixed time scaling $s=\tau/T$, defines

$$\hat x(\tau)=x_0+s\,N(s;\theta),$$

so $\hat x(0)=x_0$ **exactly for any weights**: no initial-condition loss is needed. Automatic differentiation with respect to physical time applies the chain rule $d\hat x/d\tau=T^{-1}\,d\hat x/ds$ (with the product rule on $s\,N$).

**Loss.** At $n=300$ fixed uniform collocation times $\tau_i\in[0,T]$:

$$R(\tau)=\frac{d\hat x}{d\tau}-f(\hat x),\qquad \mathcal L(\theta)=\frac1n\sum_i R(\tau_i)^2 .$$

**Algorithm.** (1) fixed collocation grid; (2) $\hat x$ and $d\hat x/d\tau$ by `jax.grad` + `vmap`; (3) $\mathcal L$; (4) Adam for 1000 iterations (about $2/7$ of the 3500-iteration budget) to reach a good region, then L-BFGS for 2500 iterations to converge. No solver solution is used as a label. For A the training interval ends at the extinction time, so the zero tail never enters the residual.

**Evaluation.** Dense DOP853 reference (rtol $10^{-12}$), separate from training. Metrics: RMSE, MAE, global relative $L^2$ (reported in scientific notation), max error, max error where $x_{\rm ref}<0.05$ (near-zero error), negative samples, runtime. A prediction containing NaN or infinity is *never scored* (it is labelled invalid and cannot count as accurate); a run that raises an exception is recorded as failed. A run is *inaccurate* if relative $L^2>5\times10^{-2}$ (threshold fixed beforehand).


In [ ]:
# Live demo (a few seconds on CPU): short run (100 Adam + 500 L-BFGS), Model B regular, seed 0, final design.
# A short budget is not the final result; compare with the curated table below.
import sys

sys.path.insert(0, "experiments/wp04")
import run as wp04

from mm_allee.pinn.train_forward import evaluate_prediction, predict

case = dict(model="B", a=0.2, harvest=0.3, x0=0.8, tau_end=10.0)
ref = dict(method="DOP853", rtol=1e-12, atol=1e-14, n_points=1001)
tau, x_ref, hit = wp04.reference_trajectory(case, ref)
training = {"activation": "swish", "n_collocation": 300, "adam_iters": 100, "lbfgs_iters": 500}
res = wp04.run_pinn(case, 0, {"training": training}, hit)
ev = evaluate_prediction(predict(res.params, res.config, tau), x_ref)
print({k: (f"{v:.3e}" if isinstance(v, float) else v) for k, v in ev.items()})
print(f"{res.wall_seconds:.1f} s, final physics loss {res.history['loss'][-1]:.3e}")

## Activation selection (tanh, swish, sigmoid)

With 300 uniform collocation points, seeds 0/1/42 and all six cases (54 runs), the activation with the most accurate runs out of 18 (ties: smaller geometric-mean error) was selected; the rule was written in `experiments/wp04/README.md` before running. The table is loaded from `artifacts/tables/wp04_activation_selection.csv`. *Caveat:* the winner is selected on the same seeds as the final sweep, so its accuracy is optimistic.

In [ ]:
import csv

for r in csv.DictReader(open("artifacts/tables/wp04_activation_selection.csv")):
    print(
        f"{r['activation']:8s} accurate {r['n_accurate']:>2s}/{r['n_runs']}  geomean rel L2 {float(r['geomean_relative_l2']):.2e}  "
        f"median {float(r['median_relative_l2']):.2e}  selected: {r['selected']}"
    )

## Curated final results (loaded, not retrained)

The table is `artifacts/tables/wp04_forward_metrics.csv`, produced by
`python experiments/wp04/run.py --config configs/wp04_final.json --output results/wp04 --artifacts artifacts` (about 1.5 min on CPU; see `artifacts/manifests/wp04_forward.json`).


In [ ]:
rows = list(csv.DictReader(open("artifacts/tables/wp04_forward_metrics.csv")))


def fmt(v):
    return f"{float(v):.2e}" if v else "-"


print(
    f"{'case':13s}{'method':7s}{'seed':5s}{'status':12s}{'rel L2':>10s}{'max err':>10s}{'near-0':>10s}{'neg':>5s}{'sec':>7s}"
)
for r in rows:
    print(
        f"{r['case']:13s}{r['method']:7s}{r['seed']:5s}{r['status']:12s}{fmt(r['relative_l2']):>10s}"
        f"{fmt(r['max_abs']):>10s}{fmt(r['max_abs_near_zero']):>10s}{r['boundary_violations']:>5s}{float(r['wall_seconds']):7.2f}"
    )

![Forward PINN vs reference](../artifacts/figures/wp04_forward.png)

![Error and loss](../artifacts/figures/wp04_error_loss.png)

Curated PNGs from the final run (EPS copies are in `artifacts/figures/`). The vertical line in the bottom row marks the Adam to L-BFGS hand-over.

## Interpretation

- **Activation:** swish had 17 of 18 accurate runs, tanh 11 and sigmoid 10 (geometric-mean relative $L^2$ $2.3\times10^{-5}$, $2.9\times10^{-4}$, $1.4\times10^{-3}$). Swish fixed the regular cases for all seeds where tanh and sigmoid failed in most. We do not know why.
- **Final sweep (swish):** 17 of 18 runs accurate (relative $L^2$ from $8\times10^{-8}$ to $4\times10^{-4}$); the one miss is A near-fold seed 0 at $5.8\times10^{-2}$, just above the $5\times10^{-2}$ threshold. A near-fold (collapse after a bottleneck) is the hardest case; reviewers' reruns on other machines and library versions failed a *different* seed, so the robust statement is that **about one in three seeds misses the threshold**. In every run L-BFGS stopped at its 2500-iteration limit (not an abnormal exit), so the iteration budget is binding; longer runs were not tried.
- Some runs predict slightly negative populations near zero (B decay seeds 0 and 42: 83 and 114 of 1001 samples, error $\le2\times10^{-5}$). Near-zero error (largest absolute error where $x_{\rm ref}<0.05$): A extinction $5.1\times10^{-8}$ to $3.1\times10^{-7}$, B decay $1.3\times10^{-6}$ to $1.8\times10^{-5}$, A near-fold $6.2\times10^{-3}$ to $1.3\times10^{-1}$.
- For Model A the horizon $T$ is the extinction time from the reference solver: the forward PINN does **not** predict the extinction time itself.
- RK45 matches the reference to about $10^{-10}$ in about $10^{-2}$ s; each PINN run took $4.8$ to $6.0$ s. For a scalar ODE the classical solver is the better tool; the PINN's value is as the engine for inverse problems (Task 5).
- *Limitations:* one parameter set per regime, three seeds, activation selected on the same seeds (then confirmed on three others, below), one machine; per-seed results depend on platform and library versions (exact versions are in the manifests). A **small training loss is not evidence of accuracy**: see the diagnostic below.


## Held-out confirmation and reproducibility

The frozen design (swish, 300 points) was run unchanged on seeds 11, 22, 33, which were **not** used to choose swish (selection used seeds 0, 1, 42). Table loaded from `artifacts/tables/wp04_confirmation_metrics.csv` (about 2 min on CPU to regenerate with `configs/wp04_confirm.json`). All 18 runs are accurate, but A near-fold seed 22 ($3.9\times10^{-2}$) is close to the threshold. Reviewers reran the design on Linux with newer libraries: 17/18 on seeds 0/1/42 with a *different* failing run, 16/18 in a third rerun, 18/18 on seeds 11/22/33. Per-seed results are therefore platform and version sensitive; the exact versions used here are in `artifacts/manifests/wp04_*_requirements.txt`.

In [ ]:
conf = [
    r
    for r in csv.DictReader(open("artifacts/tables/wp04_confirmation_metrics.csv"))
    if r["method"] == "pinn"
]
for case in dict.fromkeys(r["case"] for r in conf):
    errs = [float(r["relative_l2"]) for r in conf if r["case"] == case]
    print(f"{case:13s} seeds 11/22/33 rel L2: " + ", ".join(f"{e:.1e}" for e in errs))
print("accurate:", sum(float(r["relative_l2"]) <= 5e-2 for r in conf), "of", len(conf))

## Diagnostic: why did the regular cases fail with tanh and 100 points?

An *exploratory* experiment (`experiments/wp04/diagnose.py`; hypotheses were written in `experiments/wp04/README.md` before running) on the earlier design (tanh, 100 uniform points, seeds 11/22/33), separate from the final sweep. A regular and B regular are trained with 10 seeds under five collocation designs: `U100` (the earlier design), `U200`, `U400` (denser uniform), `P100` (100 points clustered near $\tau=0$, $\tau_i=T(i/99)^2$) and `U100_long` (10000 L-BFGS iterations). Besides the error we measure the residual $|R|$ on a dense grid *between* the training points. Command: `python experiments/wp04/diagnose.py --config configs/wp04_diagnose.json --output results/wp04_diagnose --artifacts artifacts` (about 8 min on CPU); the table below is loaded, not recomputed.


In [ ]:
for r in csv.DictReader(open("artifacts/tables/wp04_diagnose_summary.csv")):
    med = float(r["median_relative_l2"])
    print(
        f"{r['case']:10s}{r['variant']:10s} accurate {r['n_accurate']:>2s}/10  unstable-root {r['n_unstable_root']:>2s}  "
        f"zero {r['n_zero']:>2s}  negative {r['n_negative_state']:>2s}  not-converged {r['n_not_converged']:>2s}  "
        f"other {r['n_other']:>2s}  median rel L2 {med:.1e}"
    )

![Diagnostic outcomes](../artifacts/figures/wp04_diagnose.png)

**Findings (10 seeds per cell, tanh).**

- **Mechanism:** every accurate run has max dense-grid residual $\le1.2\times10^{-3}$ and every failed run $\ge8.4\times10^{-1}$, yet 45 of the 55 failures had residual $<10^{-1}$ *at the training points*. The loss at the training points cannot reveal them; a residual check on a denser grid separates all 100 runs.
- **Resolution near $\tau=0$ matters, not the point count:** denser uniform grids help only a little (accurate runs: A 3, 3, 5 and B 2, 3, 4 for 100, 200, 400 points), whereas the same 100 points clustered near $\tau=0$ gave 10/10 in both cases (median relative $L^2$ $6.5\times10^{-6}$ for A, $5.3\times10^{-6}$ for B).
- **More L-BFGS iterations do not help:** 3/10 (A) and 2/10 (B) stay accurate.
- Failures end on spurious equilibria of the ODE: the unstable root (22 runs), zero (6), the unphysical negative root of the Model A polynomial (2), or elsewhere; 10 runs had not converged (all Model A, uniform grids).
- The final design (swish, 300 uniform points) was chosen separately; tanh with 300 points still failed A regular (0/3 seeds) and B regular (1/3).
- *Caveats:* two cases only; the clustered grid was chosen after seeing the failures and has not been run in a frozen sweep over all six cases; 10 seeds cannot exclude a failure rate up to about $25$-$30\%$; outcomes are sensitive to round-off (a grid differing by one rounding unit changed one failing seed's error from $7.6\times10^{-1}$ to $9.8\times10^{-1}$).
